# 02c Align Weather To Analysis Windows

Align METAR observations to the existing V2 `analysis_window_start` windows.

Outputs:
- `adsb_airspace_eddf.slv_airspace.analysis_window_weather_context_v1`
- `adsb_airspace_eddf.obs.pipeline_run_log`

Alignment rule:
- one row per `analysis_window_start`
- latest METAR not after each window start
- bounded by `max_weather_age_minutes`


In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json
import sys
import yaml

from pyspark.sql import Window
from pyspark.sql import functions as F

if "spark" not in globals():
    raise RuntimeError("This notebook expects a Spark session, for example inside Databricks.")

UTC = timezone.utc
repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

with (repo_root / "configs" / "region_config.yaml").open("r", encoding="utf-8") as handle:
    region_config = yaml.safe_load(handle)
with (repo_root / "configs" / "pipeline_config.yaml").open("r", encoding="utf-8") as handle:
    pipeline_config = yaml.safe_load(handle)


def parse_bool(raw_value: str) -> bool:
    return raw_value.strip().lower() == "true"


def parse_float(raw_value: str) -> float:
    return float(raw_value.strip())


def parse_int(raw_value: str) -> int:
    return int(raw_value.strip())


def utc_now_naive() -> datetime:
    return datetime.now(UTC).replace(tzinfo=None)


def normalize_scalar(value):
    if value is None:
        return None
    if hasattr(value, "to_pydatetime"):
        return value.to_pydatetime()
    return value


def append_row_to_table(*, target_table: str, payload: dict) -> None:
    target_schema = spark.table(target_table).schema
    row = {field.name: normalize_scalar(payload.get(field.name)) for field in target_schema}
    spark.createDataFrame([row], schema=target_schema).write.mode("append").saveAsTable(target_table)


def format_for_id(value) -> str:
    numeric = float(value)
    if numeric.is_integer():
        return str(int(numeric))
    return str(numeric).replace('.', 'p')


def build_default_cell_scheme_id(*, focus_airport: str, horizontal_cell_nm: float, vertical_cell_ft: float, analysis_window_minutes: int, min_altitude_ft: float, max_altitude_ft: float, projection_mode: str) -> str:
    return (
        f"{focus_airport.lower()}_"
        f"h{format_for_id(horizontal_cell_nm)}nm_"
        f"v{format_for_id(vertical_cell_ft)}ft_"
        f"w{analysis_window_minutes}m_"
        f"a{format_for_id(min_altitude_ft)}to{format_for_id(max_altitude_ft)}_"
        f"{projection_mode}_v2"
    )


def resolve_latest_source_run_id(*, catalog: str):
    row = spark.sql(
        f"""
        SELECT run_id
        FROM {catalog}.obs.pipeline_run_log
        WHERE pipeline_name IN ('02b_prepare_live_states_v2', '02_clean_and_prepare_states')
          AND status = 'success'
        ORDER BY completed_at DESC
        LIMIT 1
        """
    ).first()
    if row is None:
        raise ValueError("No successful 02/02b pipeline run found. Run Silver preparation first.")
    return row["run_id"]


def resolve_latest_weather_run_id(*, catalog: str):
    row = spark.sql(
        f"""
        SELECT run_id
        FROM {catalog}.obs.ingestion_log
        WHERE pipeline_name = '01c_ingest_awc_metar'
          AND status = 'success'
        ORDER BY completed_at DESC
        LIMIT 1
        """
    ).first()
    if row is None:
        raise ValueError("No successful 01c weather ingestion run found. Run 01c_ingest_awc_metar.ipynb first.")
    return row["run_id"]


def ensure_target_table_exists(table_name: str) -> None:
    if not spark.catalog.tableExists(table_name):
        raise ValueError(f"Missing target table: {table_name}. Run 00_platform_setup_catalog_schema.ipynb first.")


def time_bounds(df, *, column_name: str):
    row = df.agg(F.min(column_name).alias("min_time"), F.max(column_name).alias("max_time")).first()
    return row["min_time"], row["max_time"]


def stringify_timestamp(value):
    return None if value is None else str(value)


focus_airport = region_config["focus_airport"]
weather_connection = pipeline_config.get("aviation_weather_connection", {})
weather_ingestion_config = pipeline_config.get("weather_ingestion", {})
default_catalog = pipeline_config.get("catalog_name", "adsb_airspace_eddf")
default_source_run_id = ""
default_weather_run_id = ""
default_cell_scheme_id = ""
default_horizontal_cell_nm = "10"
default_vertical_cell_ft = "3000"
default_analysis_window_minutes = "15"
default_min_altitude_ft = "0"
default_max_altitude_ft = "24500"
default_projection_mode = "local_nm"
default_station_id = weather_connection.get("station_id", focus_airport)
default_max_weather_age_minutes = str(weather_ingestion_config.get("alignment_max_weather_age_minutes", 180))
default_overwrite_source_run = "true"

catalog = default_catalog
source_run_id_raw = default_source_run_id
weather_run_id_raw = default_weather_run_id
cell_scheme_id_raw = default_cell_scheme_id
horizontal_cell_nm_raw = default_horizontal_cell_nm
vertical_cell_ft_raw = default_vertical_cell_ft
analysis_window_minutes_raw = default_analysis_window_minutes
min_altitude_ft_raw = default_min_altitude_ft
max_altitude_ft_raw = default_max_altitude_ft
projection_mode = default_projection_mode
station_id_raw = default_station_id
max_weather_age_minutes_raw = default_max_weather_age_minutes
overwrite_source_run_raw = default_overwrite_source_run

if "dbutils" in globals():
    dbutils.widgets.text("catalog", default_catalog)
    dbutils.widgets.text("source_run_id", default_source_run_id)
    dbutils.widgets.text("weather_run_id", default_weather_run_id)
    dbutils.widgets.text("cell_scheme_id", default_cell_scheme_id)
    dbutils.widgets.text("horizontal_cell_nm", default_horizontal_cell_nm)
    dbutils.widgets.text("vertical_cell_ft", default_vertical_cell_ft)
    dbutils.widgets.text("analysis_window_minutes", default_analysis_window_minutes)
    dbutils.widgets.text("min_altitude_ft", default_min_altitude_ft)
    dbutils.widgets.text("max_altitude_ft", default_max_altitude_ft)
    dbutils.widgets.dropdown("projection_mode", default_projection_mode, ["local_nm"])
    dbutils.widgets.text("station_id", default_station_id)
    dbutils.widgets.text("max_weather_age_minutes", default_max_weather_age_minutes)
    dbutils.widgets.dropdown("overwrite_source_run", default_overwrite_source_run, ["true", "false"])

    catalog = dbutils.widgets.get("catalog").strip() or default_catalog
    source_run_id_raw = dbutils.widgets.get("source_run_id").strip()
    weather_run_id_raw = dbutils.widgets.get("weather_run_id").strip()
    cell_scheme_id_raw = dbutils.widgets.get("cell_scheme_id").strip()
    horizontal_cell_nm_raw = dbutils.widgets.get("horizontal_cell_nm").strip() or default_horizontal_cell_nm
    vertical_cell_ft_raw = dbutils.widgets.get("vertical_cell_ft").strip() or default_vertical_cell_ft
    analysis_window_minutes_raw = dbutils.widgets.get("analysis_window_minutes").strip() or default_analysis_window_minutes
    min_altitude_ft_raw = dbutils.widgets.get("min_altitude_ft").strip() or default_min_altitude_ft
    max_altitude_ft_raw = dbutils.widgets.get("max_altitude_ft").strip() or default_max_altitude_ft
    projection_mode = dbutils.widgets.get("projection_mode").strip() or default_projection_mode
    station_id_raw = dbutils.widgets.get("station_id").strip() or default_station_id
    max_weather_age_minutes_raw = dbutils.widgets.get("max_weather_age_minutes").strip() or default_max_weather_age_minutes
    overwrite_source_run_raw = dbutils.widgets.get("overwrite_source_run").strip() or default_overwrite_source_run

selected_source_run_id = source_run_id_raw or resolve_latest_source_run_id(catalog=catalog)
selected_weather_run_id = weather_run_id_raw or resolve_latest_weather_run_id(catalog=catalog)
horizontal_cell_nm = parse_float(horizontal_cell_nm_raw)
vertical_cell_ft = parse_float(vertical_cell_ft_raw)
analysis_window_minutes = parse_int(analysis_window_minutes_raw)
min_altitude_ft = parse_float(min_altitude_ft_raw)
max_altitude_ft = parse_float(max_altitude_ft_raw)
selected_station_id = station_id_raw.upper()
max_weather_age_minutes = parse_int(max_weather_age_minutes_raw)
overwrite_source_run = parse_bool(overwrite_source_run_raw)
cell_scheme_id = cell_scheme_id_raw or build_default_cell_scheme_id(
    focus_airport=focus_airport,
    horizontal_cell_nm=horizontal_cell_nm,
    vertical_cell_ft=vertical_cell_ft,
    analysis_window_minutes=analysis_window_minutes,
    min_altitude_ft=min_altitude_ft,
    max_altitude_ft=max_altitude_ft,
    projection_mode=projection_mode,
)

silver_table_v2 = f"{catalog}.slv_airspace.flight_states_cellized_v2"
weather_table = f"{catalog}.brz_weather.metar_raw"
aligned_weather_table = f"{catalog}.slv_airspace.analysis_window_weather_context_v1"
pipeline_log_table = f"{catalog}.obs.pipeline_run_log"
for table_name in [silver_table_v2, weather_table, aligned_weather_table, pipeline_log_table]:
    ensure_target_table_exists(table_name)


In [ ]:
source_silver_df = spark.table(silver_table_v2).where(
    (F.col("run_id") == F.lit(selected_source_run_id))
    & (F.col("cell_scheme_id") == F.lit(cell_scheme_id))
)
source_row_count = source_silver_df.count()
if source_row_count == 0:
    raise ValueError(
        f"No Silver V2 rows found for run_id={selected_source_run_id} and cell_scheme_id={cell_scheme_id}."
    )

source_windows_df = source_silver_df.select("analysis_window_start").distinct()
source_window_count = source_windows_df.count()
source_window_min, source_window_max = time_bounds(source_windows_df, column_name="analysis_window_start")

weather_base_df = spark.table(weather_table).where(
    (F.col("run_id") == F.lit(selected_weather_run_id))
    & (F.col("station_id") == F.lit(selected_station_id))
)
weather_window = Window.partitionBy("station_id", "observation_time").orderBy(F.col("ingested_at").desc())
weather_df = (
    weather_base_df
    .withColumn("dedupe_rank", F.row_number().over(weather_window))
    .where(F.col("dedupe_rank") == 1)
    .drop("dedupe_rank")
)
weather_row_count = weather_df.count()
if weather_row_count == 0:
    raise ValueError(
        f"No weather rows found for weather_run_id={selected_weather_run_id} and station_id={selected_station_id}."
    )

weather_coverage_start, weather_coverage_end = time_bounds(weather_df, column_name="observation_time")
weather_expr = F.expr(f"INTERVAL {max_weather_age_minutes} MINUTES")
alignment_window = Window.partitionBy(F.col("windows.analysis_window_start")).orderBy(
    F.col("weather.observation_time").desc_nulls_last(),
    F.col("weather.ingested_at").desc_nulls_last(),
)

aligned_candidate_df = (
    source_windows_df.alias("windows")
    .join(
        weather_df.alias("weather"),
        (
            (F.col("weather.station_id") == F.lit(selected_station_id))
            & (F.col("weather.observation_time") <= F.col("windows.analysis_window_start"))
            & (F.col("weather.observation_time") >= F.col("windows.analysis_window_start") - weather_expr)
        ),
        how="left",
    )
    .withColumn("weather_rank", F.row_number().over(alignment_window))
    .where(F.col("weather_rank") == 1)
)

aligned_weather_df = (
    aligned_candidate_df
    .select(
        F.to_date(F.col("windows.analysis_window_start")).alias("window_date"),
        F.col("windows.analysis_window_start").alias("analysis_window_start"),
        F.expr(f"windows.analysis_window_start + INTERVAL {analysis_window_minutes} MINUTES").alias("analysis_window_end"),
        F.lit(selected_station_id).alias("station_id"),
        F.when(F.col("weather.observation_time").isNull(), F.lit("missing_weather")).otherwise(F.lit("aligned")).alias("alignment_status"),
        F.col("weather.observation_time").alias("observation_time"),
        F.when(
            F.col("weather.observation_time").isNull(),
            F.lit(None).cast("double"),
        ).otherwise(
            (F.unix_timestamp(F.col("windows.analysis_window_start")) - F.unix_timestamp(F.col("weather.observation_time"))) / F.lit(60.0)
        ).alias("minutes_since_observation"),
        F.lit("latest_not_after_window_start").alias("align_strategy"),
        F.col("weather.raw_text").alias("raw_text"),
        F.col("weather.report_type").alias("report_type"),
        F.col("weather.flight_category").alias("flight_category"),
        F.col("weather.temperature_c").alias("temperature_c"),
        F.col("weather.dewpoint_c").alias("dewpoint_c"),
        F.col("weather.wind_direction_deg").alias("wind_direction_deg"),
        F.col("weather.wind_speed_kt").alias("wind_speed_kt"),
        F.col("weather.wind_gust_kt").alias("wind_gust_kt"),
        F.col("weather.visibility_sm").alias("visibility_sm"),
        F.col("weather.altimeter_in_hg").alias("altimeter_in_hg"),
        F.col("weather.sea_level_pressure_mb").alias("sea_level_pressure_mb"),
        F.col("weather.ceiling_ft_agl").alias("ceiling_ft_agl"),
        F.col("weather.weather_string").alias("weather_string"),
        F.col("weather.cloud_layers_json").alias("cloud_layers_json"),
        F.col("weather.latitude").alias("latitude"),
        F.col("weather.longitude").alias("longitude"),
        F.lit(weather_table).alias("source_table"),
        F.lit(selected_weather_run_id).alias("weather_run_id"),
        F.lit(cell_scheme_id).alias("cell_scheme_id"),
        F.lit(utc_now_naive()).alias("computed_at"),
        F.lit(selected_source_run_id).alias("run_id"),
    )
)

alignment_counts = aligned_weather_df.groupBy("alignment_status").count().collect()
alignment_count_map = {row["alignment_status"]: int(row["count"]) for row in alignment_counts}
aligned_window_count = alignment_count_map.get("aligned", 0)
missing_window_count = alignment_count_map.get("missing_weather", 0)
if aligned_window_count == 0:
    raise ValueError(
        "No analysis windows could be aligned to weather observations within max_weather_age_minutes. "
        f"Weather coverage was {weather_coverage_start} to {weather_coverage_end}."
    )


In [ ]:
pipeline_started_at = utc_now_naive()
pipeline_status = "success"
pipeline_error_message = None

try:
    if overwrite_source_run:
        spark.sql(
            f"DELETE FROM {aligned_weather_table} WHERE run_id = '{selected_source_run_id}' AND cell_scheme_id = '{cell_scheme_id}' AND station_id = '{selected_station_id}'"
        )
    aligned_weather_df.write.mode("append").saveAsTable(aligned_weather_table)
except Exception as exc:
    pipeline_status = "failed"
    pipeline_error_message = f"{type(exc).__name__}: {exc}"
    raise
finally:
    append_row_to_table(
        target_table=pipeline_log_table,
        payload={
            "run_id": selected_source_run_id,
            "pipeline_name": "02c_align_weather_to_windows",
            "layer": "silver_weather_v1",
            "target_table": aligned_weather_table,
            "status": pipeline_status,
            "rows_read": source_window_count,
            "rows_written": source_window_count if pipeline_status == "success" else 0,
            "started_at": pipeline_started_at,
            "completed_at": utc_now_naive(),
            "error_message": pipeline_error_message,
            "metadata_json": json.dumps(
                {
                    "weather_run_id": selected_weather_run_id,
                    "station_id": selected_station_id,
                    "cell_scheme_id": cell_scheme_id,
                    "align_strategy": "latest_not_after_window_start",
                    "max_weather_age_minutes": max_weather_age_minutes,
                    "source_window_min": stringify_timestamp(source_window_min),
                    "source_window_max": stringify_timestamp(source_window_max),
                    "weather_coverage_start": stringify_timestamp(weather_coverage_start),
                    "weather_coverage_end": stringify_timestamp(weather_coverage_end),
                    "aligned_window_count": aligned_window_count,
                    "missing_window_count": missing_window_count,
                },
                sort_keys=True,
                default=str,
            ),
        },
    )

final_summary = {
    "status": pipeline_status,
    "source_run_id": selected_source_run_id,
    "weather_run_id": selected_weather_run_id,
    "cell_scheme_id": cell_scheme_id,
    "station_id": selected_station_id,
    "source_window_count": source_window_count,
    "source_row_count": source_row_count,
    "weather_row_count": weather_row_count,
    "aligned_window_count": aligned_window_count,
    "missing_window_count": missing_window_count,
    "max_weather_age_minutes": max_weather_age_minutes,
    "source_window_min": stringify_timestamp(source_window_min),
    "source_window_max": stringify_timestamp(source_window_max),
    "weather_coverage_start": stringify_timestamp(weather_coverage_start),
    "weather_coverage_end": stringify_timestamp(weather_coverage_end),
    "aligned_weather_table": aligned_weather_table,
}

final_summary
